In [41]:
import json
import re
from concurrent.futures import ThreadPoolExecutor
from statistics import mean
 
from anthropic import Anthropic
from dotenv import load_dotenv

In [42]:
load_dotenv()
client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-4-5"

In [43]:
client = Anthropic()
model = "claude-haiku-4-5"

In [45]:
# ---------------------------------------------------------------------------
# 1. Chat helpers (built in the "Making a request" / "Multi-turn" lessons)
# ---------------------------------------------------------------------------
 
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
 
 
def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
 
 
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send the conversation to Claude and return the reply text."""
    params = {
        "model": MODEL,
        "max_tokens": 2000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text
 
 
def _extract_json(text):
    """Pull JSON out of a reply, whether or not it is wrapped in ```json fences."""
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    return json.loads(match.group(1) if match else text)

In [46]:
# ---------------------------------------------------------------------------
# 2. The evaluator (dataset generation + model-based grading)
# ---------------------------------------------------------------------------
 
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks
 
    # -- Step A: have Claude invent test cases ------------------------------
    def generate_dataset(self, task_description, prompt_inputs_spec,
                         output_file="dataset.json", num_cases=3):
        spec = "\n".join(f'- "{k}": {v}' for k, v in prompt_inputs_spec.items())
        prompt = f"""Generate {num_cases} diverse, realistic test cases for evaluating a prompt.
 
<task_description>
{task_description}
</task_description>
 
Each test case supplies these inputs to the prompt:
{spec}
 
Return ONLY a JSON array. Each element is an object with:
- "prompt_inputs": an object with exactly the input keys listed above
- "solution_criteria": a list of 2-4 specific things a good output must do for this case
"""
        messages = []
        add_user_message(messages, prompt)
        dataset = _extract_json(chat(messages))
 
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2)
        print(f"Wrote {len(dataset)} test cases to {output_file}")
        return dataset
 
    # -- Step B: have Claude grade one output -------------------------------
    def _grade(self, test_case, output, extra_criteria):
        criteria = "\n".join(f"- {c}" for c in test_case.get("solution_criteria", []))
        extra = f"\n<extra_criteria>\n{extra_criteria}\n</extra_criteria>\n" if extra_criteria else ""
 
        prompt = f"""You are an expert reviewer grading an AI-generated output.
 
<inputs>
{json.dumps(test_case["prompt_inputs"], indent=2)}
</inputs>
 
<output_to_grade>
{output}
</output_to_grade>
 
<solution_criteria>
{criteria}
</solution_criteria>
{extra}
Score the output from 1 (fails badly) to 10 (fully meets every criterion).
Return ONLY a JSON object with keys:
"strengths" (list), "weaknesses" (list), "reasoning" (string), "score" (integer 1-10)
"""
        messages = []
        add_user_message(messages, prompt)
        return _extract_json(chat(messages, temperature=0.0))
 
    # -- Step C: run the prompt under test, then grade it -------------------
    def _run_case(self, run_prompt_function, test_case, extra_criteria):
        output = run_prompt_function(test_case["prompt_inputs"])
        grade = self._grade(test_case, output, extra_criteria)
        return {
            "test_case": test_case,
            "output": output,
            "score": grade["score"],
            "reasoning": grade["reasoning"],
            "weaknesses": grade.get("weaknesses", []),
        }
 
    def run_evaluation(self, run_prompt_function, dataset_file,
                       extra_criteria=None, output_file="results.json"):
        with open(dataset_file, encoding="utf-8") as f:
            dataset = json.load(f)
 
        # Run test cases in parallel, capped to avoid rate-limit errors
        with ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as pool:
            results = list(pool.map(
                lambda tc: self._run_case(run_prompt_function, tc, extra_criteria),
                dataset,
            ))
 
        avg = mean(r["score"] for r in results)
        print(f"Average score: {avg:.1f} / 10")
        for i, r in enumerate(results, 1):
            print(f"  Case {i}: {r['score']}/10 - {r['reasoning'][:120]}")
 
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        return results

In [50]:
%load_ext autoreload
%autoreload 2

from prompt_eval import PromptEvaluator, chat, add_user_message

evaluator = PromptEvaluator(max_concurrent_tasks=3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'prompt_eval'

In [40]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
)

NameError: name 'evaluator' is not defined